# 1. The gauge wave: a first numerical-relativity run

Flat spacetime in a wiggly time slicing,
$$ds^2 = -H\,dt^2 + H\,dx^2 + dy^2 + dz^2,\qquad H = 1 - A\sin\frac{2\pi(x-t)}{d}.$$
Space-time is flat, but the ADM variables oscillate. Because we know the exact solution, we can measure the error
and check that it converges at 4th order. This is the *Apples-with-Apples* gauge-wave test.

In [ ]:
from pynr import Simulation
import numpy as np, matplotlib.pyplot as plt

def run(dx, t_final=2.0):
    over = {"CoordBase::dx": dx, "CoordBase::dy": dx, "CoordBase::dz": dx,
            "CoordBase::ymin": -dx, "CoordBase::ymax": dx, "CoordBase::zmin": -dx, "CoordBase::zmax": dx,
            "Cactus::cctk_final_time": t_final, "IOBasic::outInfo_every": 0,
            "IO::out_dir": f"gauge_wave_dx{dx}"}
    return Simulation.from_parfile("../par/gauge_wave.par", overrides=over, verbose=False).run()

sim = run(0.02)

Compare the lapse with the exact solution along $x$:

In [ ]:
X, Y, Z = sim.grid.meshgrid()
_, _, alp_exact, _ = sim.thorn("Exact").solution(sim.time, X, Y, Z)
alp = sim.thorn("ADMBase").U[12]
j = k = sim.grid.nghost
x = sim.grid.coords1d[0]
fig, ax = plt.subplots(2, 1, sharex=True)
ax[0].plot(x, alp[:, j, k], label="pyNR"); ax[0].plot(x, alp_exact[:, j, k], "--", label="exact"); ax[0].legend()
ax[1].plot(x, alp[:, j, k] - alp_exact[:, j, k]); ax[1].set_ylabel("error"); ax[1].set_xlabel("x")
ax[0].set_title(f"t = {sim.time:.2f}")

## Convergence

Halving $\Delta x$ should reduce the error by $2^4 = 16$.

In [ ]:
errs = {}
for dx in (0.04, 0.02, 0.01):
    s = run(dx, t_final=1.0)
    X, Y, Z = s.grid.meshgrid()
    _, _, ae, _ = s.thorn("Exact").solution(s.time, X, Y, Z)
    errs[dx] = np.abs(s.thorn("ADMBase").U[12] - ae)[s.grid.interior].max()
    print(f"dx = {dx:5.3f}   max error = {errs[dx]:.3e}")
d = sorted(errs)
for a, b in zip(d[1:], d[:-1]):
    print(f"order between dx={a} and dx={b}: {np.log2(errs[a] / errs[b]):.2f}")

## Reading the output with kuibit

The output directory has the same layout as an Einstein Toolkit run.

In [ ]:
from kuibit.simdir import SimDir
sd = SimDir("gauge_wave_dx0.02")
print(sd.ts.maximum)
amax = sd.ts.maximum["alp"]
plt.plot(amax.t, amax.y); plt.xlabel("t"); plt.ylabel("max alpha")

### Exercises
1. Increase the amplitude to $A = 0.5$. How long does the run survive? (The gauge wave is famous for exposing instabilities in formulations.)
2. Switch `MoL::ODE_Method` to `"RK2"` and redo the convergence test. What order do you measure, and why?
3. Set `ADMBase::lapse_evolution_method = "1+log"`. Is the exact solution still a solution?